# ModernBERT-JEPA decider v2 (masked answer)

**Run 6, one pass through:** (1) a sanity test where the state states the answer outright, which a correctly wired model learns almost at once; (2) a fair overfit check, 200 real questions for 30 passes at a constant learning rate, with accuracy every 5 passes; (3) C1 with steadier settings (learning rate 5e-5, smoothed loss and learning rate in the log). The checks report PASS or FAIL and the run carries on either way. C2 (JEPA) stays off until C1 learns.

One question per sequence: `[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`.

- **Decision:** a fixed 38-way head at `[MASK]` over the labels A–Z, yes / no and 0–9; each question keeps only its own labels (a choice its first K letters, a statement yes / no, a score its first K digits) before the softmax. The head starts from ModernBERT's pretrained output rows for those 38 tokens and is fully trainable. This is the readout our 4B decoder uses, on an encoder.
- **JEPA (as in I-JEPA):** a target encoder, an EMA copy with no gradients, reads the same sequence with the gold label in place of `[MASK]`; a predictor maps the trained encoder's `[MASK]` vector to that target's vector at the same position.
- **Control:** the same model is trained twice from the pretrained weights: **C1** without the JEPA loss (the supervised baseline), **C2** with it.

The JEPA target carries supervised information (the gold label is in the target's input): this is a JEPA-style predictive objective with a structured target, not the unsupervised image recipe. `01_jepa_quick.ipynb` stays unchanged as the record of runs 1 and 2.

Data: **Stage 1**, a filtered sample of public typed decisions (`tasksource/tasksource-jev-typed-decisions`: commercial-use licences, nothing overlapping either benchmark); **Stage 2**, a subset of our generated data. JevBench's 231 public items are scored untrained and after each stage, never trained on.

## Parameters

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

ENCODER = env("ENCODER", "answerdotai/ModernBERT-base")
DATASET = "skamalj/signpost-d-decisions"           # our generated data (private): train and dev
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"   # pinned public items
TASKSOURCE = "tasksource/tasksource-jev-typed-decisions"       # Stage 1: public typed decisions
TS_QUESTIONS = int(env("TS_QUESTIONS", "20000"))   # Stage 1 sample size
TS_PER_SOURCE = int(env("TS_PER_SOURCE", "300"))   # at most this many from one source
TS_BUFFER = 50000                                  # streaming shuffle buffer
OVERLAP_N = 8                                      # drop a state sharing an 8-word run with a test
TRAIN_QUESTIONS = int(env("TRAIN_QUESTIONS", "6000"))  # Stage 2: a subset of our 12,274
MAX_OPTIONS = 26                                   # letters A-Z
BATCH = int(env("BATCH", "16"))                    # questions per step
TRAIN_MAX_LEN = 1024
EVAL_MAX_LEN = 8192                                # JevBench's hard items are long documents
OPTION_MAX_TOKENS = 64                             # each option's text is cut to this
EVAL_TOKEN_BUDGET = 16384                          # tokens per evaluation batch
LR_ENCODER = float(env("LR_ENCODER", "5e-5"))      # top layer; lower layers decay by LLRD
LLRD = float(env("LLRD", "0.95"))                  # gentle, so the whole encoder trains
TS_EPOCHS = int(env("TS_EPOCHS", "1"))             # passes over the Stage 1 sample
OURS_EPOCHS = int(env("OURS_EPOCHS", "3"))         # passes over our data
SANITY_TRAIN, SANITY_TEST = 300, 200               # the sanity test: the answer is in the state
OVERFIT_N, OVERFIT_EPOCHS, OVERFIT_EVERY = 200, 30, 5   # the fair overfit check
OVERFIT_LR = 5e-5                                  # constant, after a short warm-up
RUN_C2 = env("RUN_C2", "0") == "1"                 # JEPA run, off until C1 learns
LR_HEAD = float(env("LR_HEAD", "5e-4"))            # predictor, temperatures, biases
WARMUP = 0.05
EMA_START, EMA_END = 0.996, 0.9999                 # target-encoder momentum, cosine schedule
LAMBDA_JEPA, LAMBDA_ORD = 0.5, 0.5
LAMBDA_VAR = float(env("LAMBDA_VAR", "0"))         # variance regulariser on ẑ (ablation)
LAMBDA_COV = float(env("LAMBDA_COV", "0"))         # covariance term (ablation; noisy at batch 16)
SEED = int(env("SEED", "42"))

## Packages and the Hugging Face token

The token reads our private dataset: from the environment, a Kaggle or Colab secret, or a hidden prompt. Only where it came from is printed.

In [2]:
!pip install -q "transformers>=4.48"

import copy, gc, json, math, random, re, statistics, time, urllib.request
from collections import Counter, defaultdict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForMaskedLM, AutoTokenizer


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret"
    except Exception:
        pass
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token, "Colab secret"
    except Exception:
        pass
    import getpass
    token = getpass.getpass("Hugging Face token (read access): ").strip()
    return (token, "the prompt") if token else (None, "NOT FOUND")


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {token_from}")

Hugging Face token: the prompt


## Device and log

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 where the GPU has it; fp32 otherwise (ModernBERT overflows in fp16)
BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


gpu = torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU"
log(f"torch {torch.__version__} | transformers {transformers.__version__} | {gpu} | "
    f"{'bf16' if BF16 else 'fp32'} | {ENCODER}")

[09:57:32] torch 2.11.0+cu130 | transformers 5.21.0 | NVIDIA RTX PRO 6000 Blackwell Server Edition | bf16 | answerdotai/ModernBERT-base


## One item per question

Every question becomes: question text, state text, option texts and a gold distribution over the options. `noul` options are the criteria's `true` / `false` descriptions (or nothing), shown as `yes` / `no`; `choice` options are `name - description`, shown as `A`, `B`, …; `score` levels are shown as `0`, `1`, …

In [4]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def option_texts(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        return ["true", "false"], [crit.get("true") or "", crit.get("false") or ""]
    if kind == "choice":
        names = list(crit)
        return names, [f"{n} - {d}" if d else n for n, d in crit.items()]
    return [str(i) for i in range(len(crit))], [str(level) for level in crit]


def make_item(kind, question, state, q, gold, source, family=""):
    names, texts = option_texts(q)
    total = sum(gold)
    return {"type": kind, "question": question or "", "state": state_text(state), "names": names,
            "texts": texts, "gold": [g / total for g in gold], "source": source, "family": family}

## Our training and dev data

In [5]:
def load_split(split):
    path = hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN)
    items = []
    for line in open(path, encoding="utf-8"):
        rec = json.loads(line)
        for qid, q in rec["questions"].items():
            probs = rec["gold"][qid]["probabilities"]
            names, _ = option_texts(q)
            if len(names) > MAX_OPTIONS:
                continue
            gold = [float(probs[n]) for n in names]
            items.append(make_item(q["type"], q.get("instructions"), rec["state"], q, gold, split))
    return items


train_all, dev_items = load_split("train"), load_split("dev")
train_items = random.Random(SEED).sample(train_all, min(TRAIN_QUESTIONS, len(train_all)))
print(f"train {len(train_items)} of {len(train_all)} questions "
      f"{dict(Counter(it['type'] for it in train_items))} | dev {len(dev_items)}")

train.jsonl:   0%|          | 0.00/8.15M [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/661k [00:00<?, ?B/s]

train 6000 of 12274 questions {'choice': 1804, 'score': 2280, 'noul': 1916} | dev 1000


## JevBench public items (test only, never trained on)

One question per item and one expected label, so the gold is one-hot.

In [6]:
def load_jevbench(split):
    url = (f"https://raw.githubusercontent.com/fstandhartinger/jevbench/{JEVBENCH_COMMIT}"
           f"/datasets/public/{split}.jsonl")
    items = []
    for line in urllib.request.urlopen(url).read().decode("utf-8").splitlines():
        rec = json.loads(line)
        q = rec["question"]
        names, _ = option_texts(q)
        if q["type"] == "noul":
            target = "true" if rec["expected"] == "yes" else "false"
        else:
            target = str(rec["expected"])
        gold = [1.0 if n == target else 0.0 for n in names]
        assert sum(gold) == 1, rec["id"]
        items.append(make_item(q["type"], q["instructions"], rec["state"], q, gold,
                               split, rec["family"]))
    return items


jev_items = [it for split in ("original", "easy", "hard") for it in load_jevbench(split)]
print(f"JevBench public: {len(jev_items)} items {dict(Counter(it['source'] for it in jev_items))} "
      f"{dict(Counter(it['type'] for it in jev_items))}")

JevBench public: 231 items {'original': 72, 'easy': 48, 'hard': 111} {'noul': 74, 'choice': 139, 'score': 18}


## Stage 1 data: public typed decisions, filtered

A guard set of word 8-grams from every test state we care about (JevBench's public items, Typed Decisions' train and test splits). Then the tasksource stream, shuffled: rows whose `license_use` is commercial, with 2–26 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set.

In [7]:
WORD = re.compile(r"\w+")


def ngrams(text, n=OVERLAP_N):
    words = WORD.findall(text.lower())
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


guard = set()
for it in jev_items:
    guard |= ngrams(it["state"])
for split in ("train", "test"):
    path = hf_hub_download("LocalLLaMA/typed-decisions", f"all/{split}-00000-of-00001.parquet",
                           repo_type="dataset")
    for state in pd.read_parquet(path)["state"]:
        guard |= ngrams(state_text(json.loads(state)))
print(f"guard set: {len(guard):,} 8-grams")

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

all/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

guard set: 162,767 8-grams


In [8]:
def ts_item(row):
    """A tasksource row -> one item, or None if it does not fit."""
    kind, opts, target = row["kind"], row["options"], row["target"]
    if kind == "noul":
        q, gold = {"type": "noul", "criteria": None}, [target[0], 1 - target[0]]
    elif kind == "choice" and 2 <= len(opts) <= MAX_OPTIONS and len(set(opts)) == len(opts):
        q, gold = {"type": "choice", "criteria": {o: None for o in opts}}, list(target)
    elif kind == "score" and 2 <= len(opts) <= 10:
        q, gold = {"type": "score", "criteria": list(opts)}, list(target)
    else:
        return None
    if sum(gold) <= 0 or len(gold) != len(option_texts(q)[0]):
        return None
    return make_item(kind, row["question"], row["state"], q, gold, "tasksource", row["source"])


stream = load_dataset(TASKSOURCE, "default", split="train", streaming=True, token=HF_TOKEN)
stream = stream.shuffle(seed=SEED, buffer_size=TS_BUFFER)
ts_items, per_source, dropped = [], Counter(), Counter()
for row in stream:
    if row["license_use"] != "commercial":
        dropped["licence"] += 1
        continue
    if per_source[row["source"]] >= TS_PER_SOURCE:
        continue
    item = ts_item(row)
    if item is None:
        dropped["shape"] += 1
        continue
    if ngrams(item["state"]) & guard:
        dropped["overlap"] += 1
        continue
    ts_items.append(item)
    per_source[row["source"]] += 1
    if len(ts_items) >= TS_QUESTIONS:
        break
print(f"Stage 1: {len(ts_items)} questions from {len(per_source)} sources "
      f"{dict(Counter(it['type'] for it in ts_items))} | dropped {dict(dropped)}")

README.md:   0%|          | 0.00/12.8k [00:00<?, ?B/s]

Stage 1: 20000 questions from 401 sources {'choice': 15120, 'noul': 2974, 'score': 1906} | dropped {'licence': 14744, 'shape': 988, 'overlap': 8}


## Label tokens

The 38 labels are A–Z, yes / no and 0–9. Each must be one token of ModernBERT's vocabulary (checked here, with the leading space it has after `Answer:`): the head starts from those tokens' pretrained output rows, and the gold label's token fills the slot in C2's target input.

In [9]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD, MASK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id
LABELS = {"choice": [chr(65 + i) for i in range(MAX_OPTIONS)],
          "noul": ["yes", "no"],
          "score": [str(i) for i in range(10)]}


def one_token(label):
    ids = tok(" " + label, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"label {label!r} is {len(ids)} tokens"
    return ids[0]


LABEL_IDS = {kind: [one_token(lab) for lab in labs] for kind, labs in LABELS.items()}
HEAD_TOKENS = LABEL_IDS["choice"] + LABEL_IDS["noul"] + LABEL_IDS["score"]   # the 38 head outputs
HEAD_SLOTS = {"choice": list(range(26)), "noul": [26, 27], "score": list(range(28, 38))}
print({k: v[:4] for k, v in LABEL_IDS.items()}, "| head outputs", len(HEAD_TOKENS))

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

{'choice': [329, 378, 330, 399], 'noul': [4754, 642], 'score': [470, 337, 374, 495]} | head outputs 38


## One sequence per question

`[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`. The question and the options are kept whole (each option cut at `OPTION_MAX_TOKENS`); if the sequence is too long, the state's end is cut, and the cut is counted. With `gold_fill`, the gold label's token stands in place of `[MASK]` (the JEPA target's input). In training, choice options are shuffled, so a letter never carries a fixed answer.

In [10]:
def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def shuffled(item, rng):
    """A copy of a choice item with its options in a random order (gold moved with them)."""
    if item["type"] != "choice" or rng is None:
        return item
    order = list(range(len(item["names"])))
    rng.shuffle(order)
    return {**item, "names": [item["names"][i] for i in order],
            "texts": [item["texts"][i] for i in order], "gold": [item["gold"][i] for i in order]}


def sequence(item, max_len):
    """-> (token ids with [MASK], mask position, gold label token, state was cut)."""
    labels = LABELS[item["type"]]
    tail = []
    for label, text in zip(labels, item["texts"]):
        line = f"{label}: {text}".strip() if text else label
        tail += ids(line)[:OPTION_MAX_TOKENS] + [SEP]
    tail += ids("Answer:")
    q = ids(item["question"])[:256]
    s = ids(item["state"])
    room = max_len - len(q) - len(tail) - 5
    cut = len(s) > room
    s = s[:max(room, 0)]
    seq = [CLS] + q + [SEP] + s + [SEP] + tail
    pos = len(seq)
    seq += [MASK, SEP]
    best = max(range(len(item["gold"])), key=item["gold"].__getitem__)
    gold_label = LABEL_IDS[item["type"]][best]
    return seq, pos, gold_label, cut


def batch_tensors(items, max_len, gold_fill=False):
    rows = [sequence(it, max_len) for it in items]
    width = max(len(r[0]) for r in rows)
    input_ids, att, pos = [], [], []
    for seq, p, gold_label, _ in rows:
        if gold_fill:
            seq = seq[:p] + [gold_label] + seq[p + 1:]
        input_ids.append(seq + [PAD] * (width - len(seq)))
        att.append([1] * len(seq) + [0] * (width - len(seq)))
        pos.append(p)
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(pos), sum(r[3] for r in rows)

## The model

- **Encoder** (trained): ModernBERT-base; its pretrained MLM transform turns the `[MASK]` vector into the space of token outputs.
- **Label head** (trained): 38 outputs (A–Z, yes / no, 0–9), started from the pretrained output rows of those tokens; each question keeps only its own labels.
- **Predictor** (trained): an MLP with a residual from the `[MASK]` vector to ẑ.
- **Target encoder** (EMA copy of the encoder, no gradients, always in eval mode): reads the gold-filled sequence; its vector at the answer position is the JEPA target. It is used only in training.

In [11]:
TYPES = ["noul", "choice", "score"]


class MaskedDecider(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(ENCODER, attn_implementation="sdpa")
        hidden = self.mlm.config.hidden_size
        for p in self.mlm.model.embeddings.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.label_head = nn.Linear(hidden, len(HEAD_TOKENS))
        with torch.no_grad():                         # start from the pretrained output rows
            self.label_head.weight.copy_(self.mlm.decoder.weight[HEAD_TOKENS])
            if self.mlm.decoder.bias is not None:
                self.label_head.bias.copy_(self.mlm.decoder.bias[HEAD_TOKENS])
        self.target = copy.deepcopy(self.mlm.model).requires_grad_(False)

    def mask_vectors(self, input_ids, att, pos):
        h = self.mlm.model(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

    def label_logits(self, vectors, items):
        """The 38-way head at [MASK]; each question keeps its own labels -> [items, options]."""
        all38 = self.label_head(self.mlm.head(vectors)).float()
        width = max(len(it["names"]) for it in items)
        out = torch.full((len(items), width), -1e4, device=vectors.device)
        for i, it in enumerate(items):
            k = len(it["names"])
            out[i, :k] = all38[i, HEAD_SLOTS[it["type"]][:k]]
        return out

    def train(self, mode=True):
        """Train the online parts; the EMA target encoder always stays in eval mode."""
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def target_vectors(self, input_ids, att, pos):
        h = self.target(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

## Scores and losses for a batch

KL to the gold; the ordinal error for `score`; with JEPA on, 1 − cos between the predicted vector and the target's vector at the answer position. Variance and covariance regularisers are off by default (ablations).

In [12]:
def forward(model, items, max_len, jepa):
    input_ids, att, pos, cut = batch_tensors(items, max_len)
    v = model.mask_vectors(input_ids, att, pos)
    scores = model.label_logits(v, items)
    extra = {}
    if jepa:
        t_ids, t_att, t_pos, _ = batch_tensors(items, max_len, gold_fill=True)
        extra["target"] = model.target_vectors(t_ids, t_att, t_pos).float()
        extra["z"] = (v + model.predictor(v)).float()
    return scores, extra, cut


def gold_matrix(items, width):
    g = torch.zeros(len(items), width, device=DEVICE)
    for i, it in enumerate(items):
        g[i, :len(it["gold"])] = torch.tensor(it["gold"], device=DEVICE)
    return g


def losses(scores, extra, items):
    g = gold_matrix(items, scores.shape[1])
    logp = F.log_softmax(scores, dim=1)
    dec = (g * (g.clamp(min=1e-12).log() - logp)).sum(1).mean()
    parts = {"dec": dec}
    is_score = torch.tensor([it["type"] == "score" for it in items], device=DEVICE)
    if is_score.any():
        levels = torch.arange(scores.shape[1], device=DEVICE, dtype=torch.float32)
        n = torch.tensor([len(it["gold"]) - 1 for it in items], device=DEVICE).clamp(min=1)
        gap = ((logp.exp() - g) * levels).sum(1).abs() / n
        parts["ord"] = gap[is_score].mean()
    if "z" in extra:
        z = extra["z"]
        parts["jepa"] = (1 - F.cosine_similarity(z, extra["target"], dim=-1)).mean()
        zc = z - z.mean(0)
        if LAMBDA_VAR:
            parts["var"] = F.relu(1 - (zc.var(0) + 1e-4).sqrt()).mean()
        if LAMBDA_COV:
            cov = (zc.T @ zc) / max(len(items) - 1, 1)
            parts["cov"] = (cov - torch.diag(torch.diag(cov))).pow(2).sum() / z.shape[1]
    weights = {"dec": 1.0, "ord": LAMBDA_ORD, "jepa": LAMBDA_JEPA, "var": LAMBDA_VAR,
               "cov": LAMBDA_COV}
    total = sum(weights[k] * v for k, v in parts.items())
    return total, {k: v.item() for k, v in parts.items()}

## The measures

JevBench's: accuracy (argmax), multi-class Brier, top-label ECE over 10 equal-width bins (empty bins left out), ordinal MAE for `score`; and KL for our soft dev gold.

In [13]:
def measure(rows):
    n = len(rows)
    bins = defaultdict(list)
    for r in rows:
        bins[min(int(r["conf"] * 10), 9)].append(r)
    ece = 0.0
    for b in bins.values():
        right = sum(x["ok"] for x in b) / len(b)
        sure = sum(x["conf"] for x in b) / len(b)
        ece += len(b) / n * abs(right - sure)
    out = {"n": n, "acc": round(sum(r["ok"] for r in rows) / n, 4),
           "brier": round(statistics.mean(r["brier"] for r in rows), 4),
           "ece": round(ece, 4), "kl": round(statistics.mean(r["kl"] for r in rows), 4)}
    mae = [r["mae"] for r in rows if r["mae"] is not None]
    if mae:
        out["ordinal_mae"] = round(statistics.mean(mae), 3)
    return out


def row_of(it, p):
    g = it["gold"]
    top, want = max(range(len(p)), key=p.__getitem__), max(range(len(g)), key=g.__getitem__)
    mae = abs(sum(i * x for i, x in enumerate(p)) - want) if it["type"] == "score" else None
    return {"ok": top == want, "conf": p[top], "margin": None,
            "brier": sum((a - b) ** 2 for a, b in zip(p, g)),
            "kl": sum(b * math.log(b / max(a, 1e-9)) for a, b in zip(p, g) if b > 0),
            "mae": mae, "type": it["type"], "source": it["source"], "family": it["family"]}

## Evaluation

Batches by token budget (long items alone), options in their given order, the measures overall and by split, type and family, and the time per item.

In [14]:
@torch.no_grad()
def evaluate(model, items, max_len, name):
    model.eval()
    lengths = [len(sequence(it, max_len)[0]) for it in items]
    order = sorted(range(len(items)), key=lengths.__getitem__)
    rows, cut, start, t0 = [None] * len(items), 0, 0, time.perf_counter()
    while start < len(order):
        batch = [order[start]]
        while (start + len(batch) < len(order)
               and lengths[order[start + len(batch)]] * (len(batch) + 1) <= EVAL_TOKEN_BUDGET):
            batch.append(order[start + len(batch)])
        group = [items[i] for i in batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, _, c = forward(model, group, max_len, jepa=False)
        cut += c
        for i, it, p in zip(batch, group, F.softmax(scores, dim=1).tolist()):
            rows[i] = row_of(it, p[:len(it["gold"])])
        start += len(batch)
    ms = (time.perf_counter() - t0) / len(items) * 1000
    result = {"all": measure(rows), "ms_per_item": round(ms, 1), "states_cut": cut}
    for key in ("source", "type", "family"):
        values = sorted({r[key] for r in rows})
        result[key] = {v: measure([r for r in rows if r[key] == v]) for v in values}
    log(f"{name}: {result['all']} | {result['ms_per_item']} ms/item | states cut {cut}")
    for key in ("source", "type"):
        log("   " + " | ".join(f"{k} {v['acc']}" for k, v in result[key].items()))
    return result

## Optimizer, schedule and EMA

Layer-wise learning-rate decay from the top encoder layer; the MLM transform at the encoder rate; the label head and predictor at a higher rate; warm-up then cosine decay across both stages. After every step the target encoder moves towards the encoder.

In [15]:
def make_optimizer(model, total_steps, lr=None, constant=False):
    lr = lr or LR_ENCODER
    layers = model.mlm.model.layers
    groups = [{"params": [p for p in layer.parameters() if p.requires_grad],
               "lr": lr * LLRD ** (len(layers) - 1 - i)} for i, layer in enumerate(layers)]
    in_layers = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in model.mlm.parameters() if p.requires_grad and id(p) not in in_layers]
    groups.append({"params": rest, "lr": lr})
    extra = list(model.predictor.parameters()) + list(model.label_head.parameters())
    groups.append({"params": extra, "lr": LR_HEAD})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    warm = max(1, int(WARMUP * total_steps))

    def factor(s):
        if s < warm:
            return (s + 1) / warm
        if constant:
            return 1.0
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, factor)


@torch.no_grad()
def ema_update(model, step, total_steps):
    m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / total_steps) + 1) / 2
    for pt, ps in zip(model.target.parameters(), model.mlm.model.parameters()):
        pt.mul_(m).add_(ps.detach(), alpha=1 - m)
    return m

## Training one stage

`epochs` passes, reshuffled each pass, choice options shuffled per item. Every 50 steps: the decision loss averaged over the last 50 batches, the current batch's losses, the top learning rate and, with JEPA on, cos(prediction, target), the spread and norm of both, and the EMA momentum.

In [16]:
def train_stage(model, opt, sched, state, items, name, jepa, epochs=1, shuffle_options=True,
                quiet=False):
    model.train()
    recent = []
    rng = random.Random(SEED + len(items))
    t0 = time.perf_counter()
    batches = []
    for _ in range(epochs):
        order = list(range(len(items)))
        rng.shuffle(order)
        batches += [order[b:b + BATCH] for b in range(0, len(order), BATCH)]
    for chunk in batches:
        batch = [shuffled(items[i], rng if shuffle_options else None) for i in chunk]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, extra, _ = forward(model, batch, TRAIN_MAX_LEN, jepa)
        loss, parts = losses(scores, extra, batch)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        params = [p for g in opt.param_groups for p in g["params"]]
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        sched.step()
        state["step"] += 1
        m = ema_update(model, state["step"], state["total"]) if jepa else 0.0
        recent = (recent + [parts["dec"]])[-50:]
        if state["step"] % 50 == 0 and not quiet:
            terms = " ".join(f"{k} {v:.4f}" for k, v in parts.items())
            more = ""
            if jepa:
                z, t = extra["z"], extra["target"]
                cos = F.cosine_similarity(z, t, dim=-1).mean().item()
                more = (f" | cos {cos:.3f}"
                        f" | z std {z.std(0).mean().item():.3f}"
                        f" norm {z.norm(dim=-1).mean().item():.1f}"
                        f" | t std {t.std(0).mean().item():.3f}"
                        f" norm {t.norm(dim=-1).mean().item():.1f} | ema {m:.5f}")
            top_lr = max(g["lr"] for g in opt.param_groups[:-1])
            log(f"{name} step {state['step']}/{state['total']} | decision loss, last 50 "
                f"{statistics.mean(recent):.4f} | batch: {terms} | grad {norm:.2f} | "
                f"lr {top_lr:.1e}{more}")
    if not quiet:
        log(f"{name} done in {(time.perf_counter() - t0) / 60:.1f} min")

## Checks before training

On a small batch from every question type: the target encoder gets no gradients and stays in eval mode after `train()`; its outputs are deterministic; the EMA step moves it towards the encoder; the gold label never appears in the context input; every loss is finite. Then the parameter counts: the inference model (encoder, MLM transform, label head) apart from the training-only predictor and EMA copy.

In [17]:
def checks():
    model = MaskedDecider().to(DEVICE)
    sample = [next(it for it in train_items if it["type"] == k) for k in TYPES]
    sample += [next(it for it in ts_items if it["type"] == k) for k in TYPES]
    model.train()
    assert not model.target.training, "target encoder left in training mode"
    assert not any(p.requires_grad for p in model.target.parameters())
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
        scores, extra, _ = forward(model, sample, TRAIN_MAX_LEN, jepa=True)
        again = forward(model, sample, TRAIN_MAX_LEN, jepa=True)[1]["target"]
    assert torch.allclose(extra["target"], again), "target outputs are not deterministic"
    loss, parts = losses(scores, extra, sample)
    assert all(math.isfinite(v) for v in parts.values()), parts
    loss.backward()
    assert all(p.grad is None for p in model.target.parameters()), "target received gradients"
    for it in sample:
        seq, pos, gold_label, _ = sequence(it, TRAIN_MAX_LEN)
        assert seq[pos] == MASK and gold_label not in seq[pos:], "gold label in the context"
    pairs = list(zip(model.target.parameters(), model.mlm.model.parameters()))
    p_t, p_o = next((t, o) for t, o in pairs if o.requires_grad)
    with torch.no_grad():
        p_o.add_(0.01)
    before = (p_t - p_o).abs().sum().item()
    ema_update(model, 1, 10)
    after = (p_t - p_o).abs().sum().item()
    assert after < before, "EMA did not move the target towards the encoder"
    count = lambda ps: sum(p.numel() for p in ps) / 1e6
    inference = count(model.mlm.parameters()) + count(model.label_head.parameters())
    log(f"checks passed | parts {parts} | inference model {inference:.1f} M | "
        f"predictor {count(model.predictor.parameters()):.1f} M | EMA copy (training only) "
        f"{count(model.target.parameters()):.1f} M")
    del model
    gc.collect()
    torch.cuda.empty_cache()


checks()

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[09:58:00] checks passed | parts {'dec': 0.9718042612075806, 'ord': 0.21155841648578644, 'jepa': 0.3431713283061981} | inference model 149.7 M | predictor 2.4 M | EMA copy (training only) 149.0 M


## Diagnostic 1: the answer is in the state

Made-up questions where the state names the answer (a choice option by name, a statement as true or false, a score level by name). Reading it off needs only the wiring to be right: the input, the option labels, the head and the loss. Trained on 300, scored on 200 others; a correctly wired model should get close to 100%.

In [18]:
WORDS = ["amber", "basalt", "cedar", "delta", "ember", "falcon", "garnet", "harbor", "indigo",
         "juniper", "kelp", "lumen", "marble", "nectar", "onyx", "pepper", "quartz", "raven"]
LEVELS = ["none", "low", "medium", "high", "critical"]


def sanity_items(n, seed):
    rng, items = random.Random(seed), []
    for i in range(n):
        kind = TYPES[i % 3]
        if kind == "choice":
            opts = rng.sample(WORDS, rng.randint(3, 6))
            pick = rng.randrange(len(opts))
            q = {"type": "choice", "criteria": {o: f"route to {o}" for o in opts}}
            state = f"Ticket {rng.randint(100, 999)}: the reviewer picked {opts[pick]}."
            gold = [1.0 if j == pick else 0.0 for j in range(len(opts))]
            question = "Which option did the reviewer pick?"
        elif kind == "noul":
            truth = rng.random() < 0.5
            q = {"type": "noul", "criteria": None}
            said = "true" if truth else "false"
            state = f"Check {rng.randint(100, 999)}: the statement is {said}."
            gold, question = ([1.0, 0.0] if truth else [0.0, 1.0]), "Is the statement true?"
        else:
            levels = LEVELS[:rng.randint(3, 5)]
            pick = rng.randrange(len(levels))
            q = {"type": "score", "criteria": levels}
            state = f"Report {rng.randint(100, 999)}: the recorded level is {levels[pick]}."
            gold = [1.0 if j == pick else 0.0 for j in range(len(levels))]
            question = "What level is recorded?"
        items.append(make_item(kind, question, state, q, gold, "sanity"))
    return items

In [19]:
def sanity_check():
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    train, test = sanity_items(SANITY_TRAIN, SEED), sanity_items(SANITY_TEST, SEED + 1)
    total = math.ceil(len(train) / BATCH) * 3
    opt, sched = make_optimizer(model, total, lr=OVERFIT_LR, constant=True)
    train_stage(model, opt, sched, {"step": 0, "total": total}, train, "sanity", False, epochs=3)
    result = evaluate(model, test, TRAIN_MAX_LEN, "sanity, 200 unseen")
    acc = result["all"]["acc"]
    log(f"sanity check: {acc} | {'PASS' if acc >= 0.95 else 'FAIL: the wiring is wrong'}")
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return result


sanity = sanity_check()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[09:58:02] sanity step 50/57 | decision loss, last 50 0.2059 | batch: dec 0.0000 ord 0.0000 | grad 0.00 | lr 5.0e-05
[09:58:03] sanity done in 0.0 min
[09:58:03] sanity, 200 unseen: {'n': 200, 'acc': 1.0, 'brier': 0.0, 'ece': 0.0, 'kl': 0.0, 'ordinal_mae': 0.0} | 0.3 ms/item | states cut 0
[09:58:03]    sanity 1.0
[09:58:03]    choice 1.0 | noul 1.0 | score 1.0
[09:58:03] sanity check: 1.0 | PASS


## Diagnostic 2: can it memorise real questions?

200 of our training questions, 30 passes, a constant learning rate after a short warm-up, options in their given order. Accuracy on those same 200 every 5 passes: a model that can learn this readout climbs towards 100%.

In [20]:
def overfit_check():
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    items = train_items[:OVERFIT_N]
    total = math.ceil(len(items) / BATCH) * OVERFIT_EPOCHS
    opt, sched = make_optimizer(model, total, lr=OVERFIT_LR, constant=True)
    state, curve = {"step": 0, "total": total}, []
    for done in range(OVERFIT_EVERY, OVERFIT_EPOCHS + 1, OVERFIT_EVERY):
        train_stage(model, opt, sched, state, items, "overfit", False, epochs=OVERFIT_EVERY,
                    shuffle_options=False, quiet=True)
        acc = evaluate(model, items, TRAIN_MAX_LEN, f"overfit after {done} passes")["all"]["acc"]
        curve.append((done, acc))
    verdict = "PASS" if curve[-1][1] >= 0.9 else "FAIL: it cannot memorise this readout"
    log(f"overfit curve {curve} | {verdict}")
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return curve


overfit_curve = overfit_check()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[09:58:09] overfit after 5 passes: {'n': 200, 'acc': 0.715, 'brier': 0.1175, 'ece': 0.1336, 'kl': 0.2136, 'ordinal_mae': 0.416} | 1.6 ms/item | states cut 0
[09:58:09]    train 0.715
[09:58:09]    choice 0.5517 | noul 0.871 | score 0.7125
[09:58:15] overfit after 10 passes: {'n': 200, 'acc': 0.885, 'brier': 0.022, 'ece': 0.1773, 'kl': 0.049, 'ordinal_mae': 0.262} | 1.6 ms/item | states cut 0
[09:58:15]    train 0.885
[09:58:15]    choice 0.8793 | noul 0.9032 | score 0.875
[09:58:20] overfit after 15 passes: {'n': 200, 'acc': 0.92, 'brier': 0.0167, 'ece': 0.2164, 'kl': 0.0342, 'ordinal_mae': 0.242} | 1.6 ms/item | states cut 0
[09:58:20]    train 0.92
[09:58:20]    choice 0.8621 | noul 0.9355 | score 0.95
[09:58:26] overfit after 20 passes: {'n': 200, 'acc': 0.945, 'brier': 0.0104, 'ece': 0.2365, 'kl': 0.021, 'ordinal_mae': 0.253} | 1.6 ms/item | states cut 0
[09:58:26]    train 0.945
[09:58:26]    choice 0.8966 | noul 0.9677 | score 0.9625
[09:58:32] overfit after 25 passes: {'n': 200,

## One complete run

From the pretrained weights: score untrained, Stage 1, score, Stage 2, score, and accuracy on 1,000 of the questions it trained on (memorising, apart from generalising to dev).

In [21]:
def run(name, jepa):
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    total = (TS_EPOCHS * math.ceil(len(ts_items) / BATCH)
             + OURS_EPOCHS * math.ceil(len(train_items) / BATCH))
    opt, sched = make_optimizer(model, total)
    state = {"step": 0, "total": total}
    res = {"untrained": (evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench untrained"),
                         evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev untrained"))}
    train_stage(model, opt, sched, state, ts_items, f"{name} stage 1", jepa, epochs=TS_EPOCHS)
    res["stage 1"] = (evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench after stage 1"),
                      evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev after stage 1"))
    train_stage(model, opt, sched, state, train_items, f"{name} stage 2", jepa, epochs=OURS_EPOCHS)
    res["stage 2"] = (evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench after stage 2"),
                      evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev after stage 2"))
    res["train subset"] = evaluate(model, train_items[:1000], TRAIN_MAX_LEN,
                                   f"{name} 1,000 of the questions it trained on")
    return model, res

## C1: without the JEPA loss

In [22]:
model_c1, res_c1 = run("C1", jepa=False)
del model_c1
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[09:58:39] C1 JevBench untrained: {'n': 231, 'acc': 0.342, 'brier': 0.7601, 'ece': 0.1724, 'kl': 1.4194, 'ordinal_mae': 0.912} | 2.9 ms/item | states cut 0
[09:58:39]    easy 0.2917 | hard 0.4054 | original 0.2778
[09:58:39]    choice 0.2302 | noul 0.5811 | score 0.2222
[09:58:41] C1 dev untrained: {'n': 1000, 'acc': 0.219, 'brier': 0.2968, 'ece': 0.2464, 'kl': 0.5431, 'ordinal_mae': 0.847} | 1.6 ms/item | states cut 0
[09:58:41]    dev 0.219
[09:58:41]    choice 0.0867 | noul 0.46 | score 0.1375


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (14984 > 8192). Running this sequence through the model will result in indexing errors


[09:58:48] C1 stage 1 step 50/2375 | decision loss, last 50 1.3513 | batch: dec 0.9602 | grad 9.06 | lr 2.2e-05
[09:58:54] C1 stage 1 step 100/2375 | decision loss, last 50 1.3768 | batch: dec 1.3058 ord 0.1039 | grad 4.88 | lr 4.3e-05
[09:59:00] C1 stage 1 step 150/2375 | decision loss, last 50 1.3791 | batch: dec 1.7261 ord 0.0049 | grad 6.55 | lr 5.0e-05
[09:59:06] C1 stage 1 step 200/2375 | decision loss, last 50 1.3745 | batch: dec 1.0979 ord 0.2546 | grad 5.06 | lr 5.0e-05
[09:59:12] C1 stage 1 step 250/2375 | decision loss, last 50 1.2769 | batch: dec 1.6110 ord 0.2719 | grad 4.52 | lr 5.0e-05
[09:59:19] C1 stage 1 step 300/2375 | decision loss, last 50 1.3425 | batch: dec 1.2467 | grad 2.29 | lr 4.9e-05
[09:59:25] C1 stage 1 step 350/2375 | decision loss, last 50 1.3053 | batch: dec 1.4262 ord 0.5026 | grad 4.26 | lr 4.9e-05
[09:59:31] C1 stage 1 step 400/2375 | decision loss, last 50 1.2977 | batch: dec 1.6419 ord 0.1486 | grad 2.91 | lr 4.8e-05
[09:59:38] C1 stage 1 step 450/

## C2: with the JEPA loss (only with `RUN_C2 = 1`)

In [23]:
model_c2, res_c2 = run("C2", jepa=True) if RUN_C2 else (None, None)

## Side by side

Accuracy at every step for both runs, next to chance (one over the number of options).

In [24]:
def chance(items):
    return round(statistics.mean(1 / len(it["names"]) for it in items), 3)


steps = ["untrained", "stage 1", "stage 2"]
runs = [("C1", res_c1)] + ([("C2", res_c2)] if res_c2 else [])
print(f"{'set':18s} " + " ".join(f"{r + ' ' + s:>17s}" for r, _ in runs for s in steps)
      + f" {'chance':>7s}")
rows = [(f"JevBench {s}", lambda r, s=s: r[0]["source"][s]["acc"],
         [it for it in jev_items if it["source"] == s]) for s in ("original", "easy", "hard")]
rows += [("JevBench all", lambda r: r[0]["all"]["acc"], jev_items),
         ("our dev", lambda r: r[1]["all"]["acc"], dev_items)]
for name, get, sub in rows:
    cols = " ".join(f"{get(res[s]):>17.3f}" for _, res in runs for s in steps)
    print(f"{name:18s} {cols} {chance(sub):>7.3f}")
print(f"\nsanity (answer in the state, 200 unseen): {sanity['all']['acc']}")
print(f"overfit curve (passes, accuracy on the 200): {overfit_curve}")
for label, res in runs:
    print(f"{label} on 1,000 of its training questions: {res['train subset']['all']['acc']}")
    best = res["stage 2"][0]["all"]
    print(f"\n{label} JevBench after stage 2: Brier {best['brier']} | ECE {best['ece']} | "
          f"ordinal MAE {best.get('ordinal_mae')}")
    print(f"{label} by family:", {k: v["acc"] for k, v in res["stage 2"][0]["family"].items()})

set                     C1 untrained        C1 stage 1        C1 stage 2  chance
JevBench original              0.278             0.319             0.292   0.311
JevBench easy                  0.292             0.312             0.312   0.284
JevBench hard                  0.405             0.315             0.387   0.336
JevBench all                   0.342             0.316             0.342   0.318
our dev                        0.219             0.300             0.400   0.318

sanity (answer in the state, 200 unseen): 1.0
overfit curve (passes, accuracy on the 200): [(5, 0.715), (10, 0.885), (15, 0.92), (20, 0.945), (25, 0.94), (30, 0.955)]
C1 on 1,000 of its training questions: 0.369

C1 JevBench after stage 2: Brier 0.6845 | ECE 0.0307 | ordinal MAE 0.925
C1 by family: {'adequacy': 0.5, 'adversarial': 0.6667, 'ambiguous': 0.4286, 'extraction': 0.2083, 'fact': 0.5, 'intent': 0.2083, 'judge_hard': 0.5882, 'long_policy': 0.3158, 'multi_hop': 0.1111, 'ordinal': 0.25, 'policy': 0.5, 

## JEPA diagnostics (C2) on our dev

How close the prediction is to the target's answer vector, and how spread the predictions are.

In [25]:
@torch.no_grad()
def diagnostics(model, items):
    model.eval()
    cos, spread = [], []
    for b in range(0, len(items), 32):
        group = items[b:b + 32]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            _, extra, _ = forward(model, group, TRAIN_MAX_LEN, jepa=True)
        cos += F.cosine_similarity(extra["z"], extra["target"], dim=-1).tolist()
        spread.append(extra["z"].std(0).mean().item())
    log(f"cos(prediction, target at the answer) {statistics.mean(cos):.3f} | "
        f"z std {statistics.mean(spread):.3f}")


if RUN_C2:
    diagnostics(model_c2, dev_items)